# Week 5 · Q11 — Which replenishment policy fits? (gate and labelled scenario)
**Owner:** Ibrahim Khalil Umar, with Abdibasid Mohamed Ali · **Status: BLOCKED by Q10 for reportable stock KPIs.**

Q10 found purchase receipts are 18.1× units sold, so turnover, days of cover, stock-health classes and a replenishment policy cannot be *reported* until the data owner explains the purchase-order quantities.
This notebook therefore does two things and nothing more:
1. **Gate:** re-verify the Q10 scale finding on the corrected data, so the block is evidenced.
2. **Scenario (labelled, not a finding):** treat the **sales side as the reference scale**, ignore purchase quantities and `current_stock`, and test how the existing reorder thresholds behave against actual demand and supplier lead time.

Every scenario output carries the label `SCENARIO — sales-side scale assumed; not a reportable stock KPI`.

**Scenario figures (for example "5.7% of demand lost") depend on three assumptions: lost sales with no backorder, `opening_stock` as the starting stock, and a constant lead time with no variability. They must not be quoted outside the labelled scenario.**

In [1]:
import os, pathlib, math
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path.cwd()
def find_merged():
    env = os.environ.get("CADETX_MERGED")
    if env and pathlib.Path(env).exists():
        return pathlib.Path(env)
    for p in [HERE, *HERE.parents]:
        c = p / "week-02" / "reports" / "week_02" / "merged"
        if c.exists():
            return c
    raise SystemExit("Merged datasets not found: set CADETX_MERGED or run inside the repository checkout.")

MERGED = find_merged()
OUT = HERE / "q11_outputs"; CH = HERE / "q11_charts"
OUT.mkdir(exist_ok=True); CH.mkdir(exist_ok=True)
LABEL = "SCENARIO — sales-side scale assumed; not a reportable stock KPI"
ASSUMPTIONS = "lost sales with no backorder; start stock = opening_stock; constant lead time (no variability); do not quote outside this labelled scenario"
CHECKS = []
def check(name, passed, detail=""):
    CHECKS.append({"check": name, "passed": bool(passed), "detail": detail})
    print(("PASS " if passed else "FAIL ") + name + (" | " + str(detail) if detail != "" else ""))

K = ["product_id", "branch_id"]
L = pd.read_csv(MERGED / "stock_ledger_enriched.csv", low_memory=False, parse_dates=["movement_date"])
I = pd.read_csv(MERGED / "inventory_with_ledger.csv")
P = pd.read_csv(MERGED / "purchase_orders_full.csv", low_memory=False,
                usecols=["po_id", "product_id", "branch_id", "quantity", "received_date", "po_status", "lead_time_actual_days"])
print("data folder:", MERGED)

data folder: <repository>/week-02/reports/week_02/merged


## 1. Gate — is the scale gap still there?

In [2]:
tot = L.groupby("movement_type").quantity.sum()
ratio = tot["IN"] / tot["OUT"]
print(f"ledger IN {tot['IN']:,} | OUT {tot['OUT']:,} | ratio {ratio:.2f}")
check("ledger IN is about 18.1× OUT (Q10 reproduces)", abs(ratio - 18.13) < 0.1, round(ratio, 2))
check("180 product–branch pairs", len(I) == 180)

days = (L[L.movement_type == "OUT"].movement_date.max() - L[L.movement_type == "OUT"].movement_date.min()).days + 1
per_pair = L[L.movement_type == "OUT"].groupby(K).quantity.sum().reindex(pd.MultiIndex.from_frame(I[K]), fill_value=0)
monthly_sales = per_pair.values / (days / 30.4375)
monthly_in = L[L.movement_type == "IN"].groupby(K).quantity.sum().reindex(pd.MultiIndex.from_frame(I[K]), fill_value=0).values / (days / 30.4375)
gate = pd.DataFrame({"measure": [
    "ledger IN / OUT units", "median sales per pair per month (units)", "median purchase receipts per pair per month (units)",
    "median max_stock", "median reorder_level", "median safety_stock", "median current_stock",
    "median of per-pair current_stock / max_stock ratios", "median monthly receipts / max_stock",
    "median monthly sales / max_stock"],
    "value": [round(ratio, 2), round(float(np.median(monthly_sales)), 1), round(float(np.median(monthly_in)), 1),
              float(I.max_stock.median()), float(I.reorder_level.median()), float(I.safety_stock.median()), float(I.current_stock.median()),
              round(float((I.current_stock / I.max_stock).median()), 1),
              round(float(np.median(monthly_in / I.max_stock.values)), 2), round(float(np.median(monthly_sales / I.max_stock.values)), 3)]})
extra = pd.DataFrame({"measure": ["ratio of medians: median current_stock / median max_stock"],
                      "value": [round(float(I.current_stock.median() / I.max_stock.median()), 1)]})
gate = pd.concat([gate, extra], ignore_index=True)
gate.to_csv(OUT / "q11_scale_gate.csv", index=False)
print(gate.to_string(index=False))
check("monthly sales per pair is on the same order as the thresholds (median 80 to 110 units)", 80 <= np.median(monthly_sales) <= 110, round(float(np.median(monthly_sales)), 1))
check("current_stock is more than 200× max_stock (median of per-pair ratios about 349×; ratio of medians about 347×)", (I.current_stock / I.max_stock).median() > 200 and I.current_stock.median() / I.max_stock.median() > 200, f"{(I.current_stock / I.max_stock).median():.1f} / {I.current_stock.median() / I.max_stock.median():.1f}")

ledger IN 20,397,795 | OUT 1,124,832 | ratio 18.13
PASS ledger IN is about 18.1× OUT (Q10 reproduces) | 18.13
PASS 180 product–branch pairs


                                                  measure      value
                                    ledger IN / OUT units     18.130
                  median sales per pair per month (units)     84.300
      median purchase receipts per pair per month (units)   1572.900
                                         median max_stock    310.500
                                     median reorder_level     62.000
                                      median safety_stock     43.500
                                     median current_stock 107790.500
      median of per-pair current_stock / max_stock ratios    349.400
                      median monthly receipts / max_stock      5.090
                         median monthly sales / max_stock      0.277
ratio of medians: median current_stock / median max_stock    347.200
PASS monthly sales per pair is on the same order as the thresholds (median 80 to 110 units) | 84.3
PASS current_stock is more than 200× max_stock (median of per-pair ratios

## 1b. Why 84.3 units a month here, against about 95 in Week 4?
Two separate reasons, checked below: (1) the ledger holds only about 91% of delivered sales lines, and (2) Week 4's figure is the **mean** across pairs, this notebook's is the **median**.

In [3]:
S = pd.read_csv(MERGED / "sales_orders_full.csv", low_memory=False, usecols=["so_id", "product_id", "branch_id", "quantity", "delivery_date", "order_status"], parse_dates=["delivery_date"])
Dv = S[S.order_status == "Delivered"]
Oo = L[L.movement_type == "OUT"]
span = (Oo.movement_date.max() - Oo.movement_date.min()).days + 1
months = span / 30.4375
idx = pd.MultiIndex.from_frame(I[K])
led_pair = Oo.groupby(K).quantity.sum().reindex(idx, fill_value=0).values / months
sal_pair = Dv.groupby(K).quantity.sum().reindex(idx, fill_value=0).values / months
cover = Oo.quantity.sum() / Dv.quantity.sum()
bridge = pd.DataFrame({"step": ["ledger OUT, median per pair per month (this notebook)", "delivered sales lines, median per pair per month (same span)",
                                 "delivered sales lines, mean per pair per month (Week 4 style)"],
                       "units_per_pair_per_month": [np.median(led_pair), np.median(sal_pair), sal_pair.mean()]})
bridge["units_per_pair_per_month"] = bridge.units_per_pair_per_month.round(1)
bridge.to_csv(OUT / "q11_demand_bridge_week4_vs_ledger.csv", index=False)
print(f"ledger OUT rows {len(Oo):,} of {len(Dv):,} delivered lines | units coverage {cover:.4f}")
print(bridge.to_string(index=False))
check("ledger OUT holds about 91% of delivered units and lines", abs(cover - 0.911) < 0.005 and abs(len(Oo) / len(Dv) - 0.911) < 0.005, f"units {cover:.4f}, lines {len(Oo) / len(Dv):.4f}")
check("coverage explains the ledger median: 84.3 / 0.911 is within 1 unit of the delivered-lines median", abs(np.median(led_pair) / cover - np.median(sal_pair)) < 1, f"{np.median(led_pair) / cover:.1f} vs {np.median(sal_pair):.1f}")
check("Week 4's about 95 is the mean across pairs on delivered lines (94.5 to 96)", 94.5 <= sal_pair.mean() <= 96, round(float(sal_pair.mean()), 1))

ledger OUT rows 107,165 of 117,656 delivered lines | units coverage 0.9108
                                                         step  units_per_pair_per_month
        ledger OUT, median per pair per month (this notebook)                      84.3
 delivered sales lines, median per pair per month (same span)                      92.7
delivered sales lines, mean per pair per month (Week 4 style)                      94.8
PASS ledger OUT holds about 91% of delivered units and lines | units 0.9108, lines 0.9108
PASS coverage explains the ledger median: 84.3 / 0.911 is within 1 unit of the delivered-lines median | 92.6 vs 92.7
PASS Week 4's about 95 is the mean across pairs on delivered lines (94.5 to 96) | 94.8


## 2. Scenario inputs: daily demand and lead time per pair
- **Demand:** ledger `OUT` movements per product–branch pair per day, over the full calendar span. (The ledger covers about 91% of delivered sales lines.)
- **Lead time:** mean `lead_time_actual_days` of received purchase-order lines for the pair (supplier contract days plus delay; Q8 showed every product is bought from every supplier).
- **Thresholds under test:** `reorder_level` (s) and `max_stock` (S) from `inventory_master`.

In [4]:
O = L[L.movement_type == "OUT"]
dates = pd.date_range(O.movement_date.min(), O.movement_date.max(), freq="D")
dem = O.groupby(K + ["movement_date"]).quantity.sum().unstack("movement_date").reindex(columns=dates, fill_value=0).fillna(0)
dem = dem.reindex(pd.MultiIndex.from_frame(I[K]), fill_value=0)
D = dem.values.astype(float)
T = D.shape[1]
check("demand matrix total equals ledger OUT total", int(D.sum()) == int(O.quantity.sum()), int(D.sum()))
check("demand matrix has 180 pairs and a full daily calendar", D.shape == (180, len(dates)), D.shape)

Pr = P[P.po_status.eq("Received") & P.lead_time_actual_days.notna()]
lt = Pr.groupby(K).lead_time_actual_days.mean().reindex(pd.MultiIndex.from_frame(I[K]))
check("every pair has a lead time from received purchase-order lines", lt.notna().all(), int(lt.notna().sum()))
Lead = np.maximum(1, np.round(lt.values)).astype(int)
print("lead time (days): min", Lead.min(), "median", int(np.median(Lead)), "max", Lead.max())
check("lead times are in a plausible range (5 to 35 days)", Lead.min() >= 5 and Lead.max() <= 35, f"{Lead.min()}–{Lead.max()}")

mu = D.mean(axis=1); sd = D.std(axis=1, ddof=1)

PASS demand matrix total equals ledger OUT total | 1124832
PASS demand matrix has 180 pairs and a full daily calendar | (180, 2203)
PASS every pair has a lead time from received purchase-order lines | 180
lead time (days): min 17 median 18 max 19
PASS lead times are in a plausible range (5 to 35 days) | 17–19


## 3. Reorder point implied by demand and lead time

In [5]:
def rop(z):
    return np.ceil(mu * Lead + z * sd * np.sqrt(Lead))
base = pd.DataFrame({"product_id": I.product_id, "branch_id": I.branch_id,
                     "mean_daily_demand": mu.round(3), "sd_daily_demand": sd.round(3), "lead_time_days": Lead,
                     "lead_time_demand": (mu * Lead).round(1), "reorder_level": I.reorder_level, "safety_stock": I.safety_stock,
                     "max_stock": I.max_stock, "rop_z1_645": rop(1.645)})
base["reorder_level_minus_lead_time_demand"] = base.reorder_level - base.lead_time_demand
base["reorder_level_covers_lead_time_demand"] = base.reorder_level >= base.lead_time_demand
base["reorder_level_covers_rop_95"] = base.reorder_level >= base.rop_z1_645
print("pairs where reorder_level >= mean demand during lead time:", int(base.reorder_level_covers_lead_time_demand.sum()), "of 180")
print("pairs where reorder_level >= 95% reorder point:", int(base.reorder_level_covers_rop_95.sum()), "of 180")
print("median ratio reorder_level / lead-time demand:", round(float((base.reorder_level / base.lead_time_demand).median()), 2))

pairs where reorder_level >= mean demand during lead time: 114 of 180
pairs where reorder_level >= 95% reorder point: 23 of 180
median ratio reorder_level / lead-time demand: 1.26


## 4. Simulation: (s, S) policy replayed on actual daily demand
Daily review; order-up-to S whenever inventory position ≤ s; the order arrives after the pair's lead time; unmet demand is lost (no backorder). Start stock = `opening_stock`.
- **Policy A (existing):** s = `reorder_level`, S = `max_stock`.
- **Policy B (scenario):** s = demand-based reorder point at 95% service; S = s + (max_stock − reorder_level), so the order size stays the same and only the trigger changes.
Sensitivity at z = 1.28 (90%) and z = 2.33 (99%).

In [6]:
def simulate(Drow, open_stock, s, S, lead):
    T_ = len(Drow); pipe = np.zeros(T_ + lead + 2); onhand = float(open_stock); on_order = 0.0
    unmet = 0.0; stock_days = 0; area = 0.0; orders = 0
    for t in range(T_):
        onhand += pipe[t]; on_order -= pipe[t]
        d = Drow[t]; sold = min(d, onhand); unmet += d - sold
        if d > sold: stock_days += 1
        onhand -= sold
        pos = onhand + on_order
        if pos <= s:
            q = S - pos
            if q > 0:
                pipe[t + lead] += q; on_order += q; orders += 1
        area += onhand
    return unmet, stock_days, area / T_, orders

def run(policy_name, s_vec, S_vec):
    rows = []
    for i in range(len(I)):
        unmet, sd_, avg, n = simulate(D[i], I.opening_stock.iloc[i], s_vec[i], S_vec[i], Lead[i])
        rows.append({"product_id": I.product_id.iloc[i], "branch_id": I.branch_id.iloc[i], "policy": policy_name,
                     "s": s_vec[i], "S": S_vec[i], "total_demand": D[i].sum(), "unmet_demand": unmet,
                     "fill_rate": 1 - unmet / D[i].sum(), "stockout_days": sd_, "avg_on_hand": avg, "orders_placed": n})
    return pd.DataFrame(rows)

s_A = I.reorder_level.values.astype(float); S_A = I.max_stock.values.astype(float)
res = [run("A existing (reorder_level, max_stock)", s_A, S_A)]
for z, nm in [(1.28, "B z=1.28 (90%)"), (1.645, "B z=1.645 (95%)"), (2.33, "B z=2.33 (99%)")]:
    s_B = rop(z); res.append(run(nm, s_B, s_B + (S_A - s_A)))
R = pd.concat(res, ignore_index=True)
R["label"] = LABEL
R["assumptions"] = ASSUMPTIONS
R.to_csv(OUT / "q11_scenario_pair_results.csv", index=False)

summ = R.groupby("policy").agg(pairs=("product_id", "size"), total_demand=("total_demand", "sum"), unmet_demand=("unmet_demand", "sum"),
                               stockout_days=("stockout_days", "sum"), avg_on_hand_per_pair=("avg_on_hand", "mean"),
                               orders_placed=("orders_placed", "sum"), pairs_with_fill_below_95=("fill_rate", lambda x: int((x < .95).sum()))).reset_index()
R = R.merge(I[K + ["unit_cost"]], on=K)
R["avg_on_hand_value_at_cost"] = R.avg_on_hand * R.unit_cost
R.to_csv(OUT / "q11_scenario_pair_results.csv", index=False)
summ = summ.merge(R.groupby("policy").avg_on_hand_value_at_cost.sum().rename("avg_on_hand_value_at_cost_total").reset_index(), on="policy")
summ["fill_rate"] = 1 - summ.unmet_demand / summ.total_demand
summ["stockout_day_share"] = summ.stockout_days / (len(I) * T)
summ["label"] = LABEL
summ["assumptions"] = ASSUMPTIONS
summ.to_csv(OUT / "q11_scenario_summary.csv", index=False)
base["label"] = LABEL
base["assumptions"] = ASSUMPTIONS
base.to_csv(OUT / "q11_scenario_reorder_points.csv", index=False)
print(summ[["policy", "fill_rate", "stockout_day_share", "avg_on_hand_per_pair", "avg_on_hand_value_at_cost_total", "orders_placed", "pairs_with_fill_below_95"]].round(4).to_string(index=False))

check("simulation: sold + unmet equals demand for every policy and pair",
      np.allclose(R.total_demand - R.unmet_demand, R.total_demand * R.fill_rate), "")
check("4 policies x 180 pairs simulated", len(R) == 720, len(R))
check("fill rate lies between 0 and 1 everywhere", R.fill_rate.between(0, 1).all())
check("higher service factor never lowers total fill rate (B90 <= B95 <= B99)",
      summ.set_index("policy").loc["B z=1.28 (90%)", "fill_rate"] <= summ.set_index("policy").loc["B z=1.645 (95%)", "fill_rate"] + 1e-9 <=
      summ.set_index("policy").loc["B z=2.33 (99%)", "fill_rate"] + 2e-9)

                               policy  fill_rate  stockout_day_share  avg_on_hand_per_pair  avg_on_hand_value_at_cost_total  orders_placed  pairs_with_fill_below_95
A existing (reorder_level, max_stock)     0.9426              0.0140              152.1364                     3.925898e+08           4457                        72
                       B z=1.28 (90%)     0.9891              0.0028              170.3423                     4.413722e+08           4804                         0
                      B z=1.645 (95%)     0.9939              0.0016              180.2971                     4.683394e+08           4839                         0
                       B z=2.33 (99%)     0.9981              0.0005              199.7031                     5.190848e+08           4872                         0
PASS simulation: sold + unmet equals demand for every policy and pair
PASS 4 policies x 180 pairs simulated | 720
PASS fill rate lies between 0 and 1 everywhere
PASS higher se

## 5. Charts

In [7]:
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.8))
ax[0].scatter(base.rop_z1_645, base.reorder_level, s=14, alpha=.6, color="#16213A")
m = max(base.rop_z1_645.max(), base.reorder_level.max()) * 1.05
ax[0].plot([0, m], [0, m], color="#999", ls="--", lw=.8)
ax[0].set_xlabel("Demand-based reorder point (95%), units"); ax[0].set_ylabel("Existing reorder_level, units")
ax[0].set_title("Existing trigger vs demand-based trigger, 180 pairs")
sm = summ.set_index("policy")
names = list(sm.index)
ax[1].bar(range(len(names)), sm.fill_rate * 100, color=["#C8791A"] + ["#1B6B66"] * 3)
ax[1].set_xticks(range(len(names))); ax[1].set_xticklabels(["A existing", "B 90%", "B 95%", "B 99%"])
ax[1].set_ylim(max(0, sm.fill_rate.min() * 100 - 6), 100.5); ax[1].set_ylabel("Fill rate, %")
for i, v in enumerate(sm.fill_rate * 100): ax[1].text(i, v + .1, f"{v:.1f}%", ha="center")
ax[1].set_title("Scenario fill rate (lost sales when out of stock)")
fig.text(0.5, -0.02, LABEL + " | lost sales, opening-stock start, constant lead time", ha="center", fontsize=9, color="#555")
plt.tight_layout(); plt.savefig(CH / "q11_01_reorder_points_and_fill_rate.png", dpi=150, bbox_inches="tight"); plt.close()
check("chart written", (CH / "q11_01_reorder_points_and_fill_rate.png").exists())

PASS chart written


## 6. Validation

In [8]:
V = pd.DataFrame(CHECKS)
V.to_csv(HERE / "validation_report_week05_q11.csv", index=False)
print(V.passed.sum(), "of", len(V), "checks pass")
assert V.passed.all(), V[~V.passed]

16 of 16 checks pass
